# mjorbit P4.4d: ACT on demos_v4 (LeRobot 0.6.1)

Behaviour cloning of the classical grasp expert (SPEC_4 §4.4). Data: 125 clean plus 125 DART episodes on the realistic arm (arm2), images at 10 Hz. Inputs are the wrist, bus (128 px) and lidar (grey range) images plus the 29-D arm state. The output is a chunk of 20 grip targets (bus frame, position plus rot6) and the close flag. 220 episodes train; the last 15 of each source are validation.

Before running: **Runtime > Change runtime type > L4 GPU** (preferred: Ada, 24 GB, more vCPUs for video decoding; a T4 also works, slower). Drive layout: `MyDrive/mjorbit_p4/data_v4/` (`demos_v4_lerobot.tar` plus `SHA256SUMS`) and `MyDrive/mjorbit_p4/code_v4/` (`val_act.py`). Checkpoints and logs go to `MyDrive/mjorbit_p4/act_v4/run1/`, every 2000 steps.

If the session disconnects, run all cells again: training resumes from the last checkpoint on Drive. To stop early (for example if validation is flat), interrupt cell 5 and run cells 6 and 7.

**Fresh runtime (13 Nov):** start from **Runtime > Disconnect and delete runtime**, choose L4, then **Run all**. Cell 3 installs only LeRobot on Colab's own CUDA stack and removes `torchaudio` (not used; a CUDA build mismatch with torch breaks the `transformers` import that LeRobot needs). It stops the notebook if the import check fails, so training never starts on a broken install.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!python --version; nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/mjorbit_p4'
OUT = f'{D}/act_v4/run1'
!ls -la {D}/data_v4 {D}/code_v4

In [ ]:
# LeRobot 0.6.1 (Python >= 3.12, torch 2.7-2.11) on Colab's own CUDA stack; pip keeps Colab's torch if it
# fits, else installs the default wheel. torchaudio is removed: LeRobot does not use it, and a torchaudio
# built for another CUDA than torch makes transformers' AutoProcessor import fail (first Colab run, 13 Nov).
import sys, subprocess
assert sys.version_info >= (3, 12), f'LeRobot 0.6.1 needs Python >= 3.12, this runtime has {sys.version}'
!pip install -q 'lerobot[training]==0.6.1'
!pip uninstall -y -q torchaudio
import importlib.metadata as m
for k in ('lerobot', 'torch', 'torchvision', 'transformers'):
    print(k, m.version(k))
chk = subprocess.run([sys.executable, '-c', 'import torch; from transformers import AutoProcessor; '
                      'from lerobot.policies.act.modeling_act import ACTPolicy; '
                      'print("torch cuda", torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0)); '
                      'print((torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")).shape)'],
                     capture_output=True, text=True)
print(chk.stdout[-2000:], chk.stderr[-3000:])
assert chk.returncode == 0, 'install check failed: see the error above (paste it to Claude)'
print('install OK')


In [ ]:
# copy the dataset to the local disk (reading videos through the Drive mount is slow), check, unpack
import os
if not os.path.exists('/content/data/demos_v4/meta/info.json'):
    !mkdir -p /content/data && cp {D}/data_v4/demos_v4_lerobot.tar {D}/data_v4/SHA256SUMS /content/data/
    !cd /content/data && sha256sum -c SHA256SUMS && tar -xf demos_v4_lerobot.tar && rm demos_v4_lerobot.tar
!du -sh /content/data/demos_v4; ls /content/data
!mkdir -p /content/code && cp {D}/code_v4/*.py /content/code/

In [ ]:
# train (resumes from the last Drive checkpoint if there is one). 20k steps x batch 32 = 640k samples,
# about 6 passes over the 103k training frames. ACT defaults otherwise (ResNet-18 ImageNet, kl_weight 10, lr 1e-5).
import json, os, subprocess, torch
NW = max(2, min(8, os.cpu_count() - 1))  # video-decoding data workers: 2 on a T4 runtime, more on an L4
print('GPU', torch.cuda.get_device_name(0), 'data workers', NW)
EPS = json.load(open('/content/data/demos_v4_episodes.json'))
train_eps = '[' + ','.join(str(e['episode']) for e in EPS if e['split'] == 'train') + ']'
last = f'{OUT}/checkpoints/last/pretrained_model/train_config.json'
if os.path.exists(last):
    cmd = ['lerobot-train', f'--config_path={last}', '--resume=true']
else:
    if os.path.isdir(OUT):  # a failed start leaves the folder without a checkpoint; lerobot-train refuses it
        import shutil; shutil.rmtree(OUT)
    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    cmd = ['lerobot-train', '--policy.type=act', '--policy.device=cuda', '--policy.push_to_hub=false',
           '--policy.chunk_size=20', '--policy.n_action_steps=20', '--policy.use_amp=true',
           '--dataset.repo_id=local/mjorbit_demos_v4', '--dataset.root=/content/data/demos_v4',
           f'--dataset.episodes={train_eps}',
           '--batch_size=32', '--steps=20000', f'--num_workers={NW}', '--log_freq=200', '--save_freq=2000',
           '--wandb.enable=false', f'--output_dir={OUT}', '--job_name=act_v4_run1']
print(' '.join(cmd[:4]), '...')
with open('/content/train.log', 'a') as log:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        log.write(line)
        if 'step:' in line or 'Error' in line or 'error:' in line or 'End of training' in line:
            print(line.rstrip()[-220:], flush=True)
    p.wait()
!mkdir -p {OUT} && cp /content/train.log {OUT}/train_colab_$(date +%s).log


In [ ]:
# validation: offline action error of every saved checkpoint on the 30 validation episodes
# (every 5th frame): grip-target position mm and rotation deg per phase, close-flag accuracy
import glob
for ck in sorted(glob.glob(f'{OUT}/checkpoints/0*/pretrained_model')):
    step = ck.split('/')[-2]
    res = f'{OUT}/val_{step}.json'
    if not os.path.exists(res):
        print('==', step, flush=True)
        !cd /content/code && python val_act.py {ck} /content/data/demos_v4 --every 5 --device cuda --out {res} 2>&1 | grep -E '^ +(all|settle|reach|approach|close|hold):'

In [ ]:
# summary per checkpoint: median position error mm (all, reach, approach), close accuracy at close
for f in sorted(glob.glob(f'{OUT}/val_0*.json')):
    r = json.load(open(f))
    print(f.split('val_')[-1][:-5], *(round(r[k]['pos_mm_med'], 1) for k in ('all', 'reach', 'approach')),
          round(r.get('close', {}).get('close_acc', float('nan')), 3))